# MKVCodec Python入門

このNotebookでは、初めてMKVCodecを使う人向けに、動画の作成・情報取得・デコード・NumPy画像処理・再エンコードを順番に試します。最後に、CPUのcopyを減らす方法とGPU-resident処理の入口も紹介します。

**このNotebookでできるようになること**

1. 利用可能なbackendとcodecを確認する
2. NumPy画像からVP9動画を作る
3. codec・解像度・fps・フレーム数を調べる
4. OpenCVに近い感覚で動画を読む
5. NumPyで画像処理して再エンコードする
6. prefetch、thread、borrowed frame、metricsを理解する
7. 対応GPUがある場合にGPU surfaceをdecodeからencodeへ渡す

## 0. 前提と実行方法

リポジトリのroot、またはこのNotebookを開いた状態で上から順に実行してください。正式wheelの公開前はnative libraryを先にbuildします。

```shell
cmake --preset default
cmake --build --preset default
```

wheelをinstall済みならそのpackageを使います。source treeから実行する場合、次のセルが代表的なbuild先からnative libraryを探します。見つからなければ、環境変数 `MKVC_LIBRARY_PATH`へDLL/soの絶対pathを設定してください。

In [ ]:
from pathlib import Path
import os
import sys

project_root = Path.cwd().resolve()
if not (project_root / 'python' / 'mkvcodec').exists():
    if (project_root.parent / 'python' / 'mkvcodec').exists():
        project_root = project_root.parent

source_package = project_root / 'python'
if source_package.exists():
    sys.path.insert(0, str(source_package))

if 'MKVC_LIBRARY_PATH' not in os.environ:
    candidates = [
        project_root / 'build' / 'windows-all-backends' / 'Release' / 'mkvcodec.dll',
        project_root / 'build' / 'default' / 'Release' / 'mkvcodec.dll',
        project_root / 'build' / 'default' / 'libmkvcodec.so',
        project_root / 'build' / 'intel' / 'libmkvcodec.so',
    ]
    native = next((path for path in candidates if path.exists()), None)
    if native is not None:
        os.environ['MKVC_LIBRARY_PATH'] = str(native)

try:
    import mkvcodec
except ImportError as error:
    raise RuntimeError(
        'mkvcodecを読み込めません。native libraryをbuildし、'
        'MKVC_LIBRARY_PATHを設定してください。'
    ) from error

import numpy as np

output_dir = project_root / 'build' / 'tutorial-output'
output_dir.mkdir(parents=True, exist_ok=True)
print('mkvcodec version:', mkvcodec.__version__)
print('native library:', os.environ.get('MKVC_LIBRARY_PATH', 'wheel同梱library'))
print('出力先:', output_dir)

## 1. 利用可能なbackendを確認する

`backend_capabilities()`は、現在のlibrary・GPU・driverで利用できるcodec方向を返します。GPU版をbuildしていても、実行環境が対応しない機能は`False`になります。

In [ ]:
capabilities = mkvcodec.backend_capabilities()
for item in capabilities:
    print(
        f'backend={item.backend:7s} codec={item.codec:3s} '
        f'decode={str(item.can_decode):5s} encode={str(item.can_encode):5s} '
        f'hardware={item.is_hardware}'
    )

## 2. 練習用のNumPyフレームを作る

入力動画がなくても試せるように、色が変化しながら白い四角形が移動するBGR画像を作ります。OpenCVと同じくshapeは `(height, width, 3)`、dtypeは`uint8`、channel順はBGRです。codec都合で幅と高さは偶数にします。

In [ ]:
width, height = 320, 180
fps = 30
frame_count = 90
yy, xx = np.indices((height, width))

def make_frame(index: int) -> np.ndarray:
    '''練習用の連続BGRフレームを1枚作ります。'''
    frame = np.empty((height, width, 3), dtype=np.uint8)
    frame[..., 0] = (xx + index * 3) % 256
    frame[..., 1] = (yy * 2 + index * 2) % 256
    frame[..., 2] = (index * 5) % 256
    left = (index * 4) % (width - 40)
    frame[70:110, left:left + 40] = (255, 255, 255)
    return frame

sample_frame = make_frame(0)
print(sample_frame.shape, sample_frame.dtype, sample_frame.flags.c_contiguous)

matplotlibがあれば画像を表示します。なくても以降のencode/decodeには影響しません。OpenCVを使う場合、返されたBGR配列をそのまま`cv2`へ渡せます。

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 4))
    plt.imshow(sample_frame[..., ::-1])  # matplotlib表示時だけBGR→RGB
    plt.axis('off')
    plt.show()
except ImportError:
    print('matplotlib未導入のため表示を省略します。フレーム生成は成功しています。')

## 3. VP9動画へencodeする

`VideoWriter`へfps、frame size、codecを渡します。`queue_size=4`は最大4件までnative workerへ先行投入する設定です。通常の`write()`はqueueに空きができるまで待ち、context managerを抜けるとflush/closeします。

In [ ]:
source_video = output_dir / 'tutorial_source.webm'

with mkvcodec.VideoWriter(
    source_video, codec='vp9', backend='cpu', fps=fps,
    frame_size=(width, height), quality=32, queue_size=4,
) as writer:
    for index in range(frame_count):
        writer.write(make_frame(index))

print('作成しました:', source_video)
print('size(bytes):', source_video.stat().st_size)
print('encode metrics:', writer.metrics)

## 4. decodeせず動画情報を調べる

`probe_video()`は画素をdecodeせず、codec、解像度、fps、duration、frame countを取得します。入力codecを手作業で調べる必要はありません。

In [ ]:
info = mkvcodec.probe_video(source_video)
print('codec      :', info.codec)
print('size       :', (info.width, info.height))
print('fps        :', info.fps)
print('duration(s):', None if info.duration_ns is None else info.duration_ns / 1e9)
print('frames     :', info.frame_count)

## 5. OpenCVに近い感覚で読む

`capture.read()`はBGRのowned NumPy配列を返し、EOSでは`None`になります。`codec='auto'`がcontainer内のVP9/AV1を判別します。owned配列はCaptureを閉じた後も安全に保持できます。

In [ ]:
with mkvcodec.VideoCapture(
    source_video, codec='auto', backend='cpu', prefetch=4
) as capture:
    first = capture.read()
    print('detected codec:', capture.codec)
    print('shape/dtype   :', first.shape, first.dtype)
    print('first PTS(ns) :', capture.last_pts_ns)

print('Capture close後の平均BGR値:', first.mean(axis=(0, 1)))

## 6. NumPyで処理して再エンコードする

ここではBGR値を反転し、緑色の枠を付けます。実アプリではこの部分をOpenCV、NumPy、画像認識処理などへ置き換えます。元動画の`info`を使えば解像度やfpsを別手段で調べる必要はありません。

In [ ]:
processed_video = output_dir / 'tutorial_processed.webm'
output_fps = info.fps if info.fps is not None else 30

with mkvcodec.VideoCapture(source_video, codec='auto', prefetch=4) as capture:
    with mkvcodec.VideoWriter(
        processed_video, codec='vp9', fps=output_fps,
        frame_size=(capture.width, capture.height), queue_size=4,
    ) as writer:
        processed_count = 0
        while (frame := capture.read()) is not None:
            result = np.ascontiguousarray(255 - frame)
            result[:4, :] = (0, 255, 0)
            result[-4:, :] = (0, 255, 0)
            result[:, :4] = (0, 255, 0)
            result[:, -4:] = (0, 255, 0)
            writer.write(result)
            processed_count += 1

print('出力:', processed_video)
print('処理フレーム数:', processed_count)
print('decode copy path:', capture.metrics.copy_path)
print('encode copy path:', writer.metrics.copy_path)

## 7. `threads`、`prefetch`、`conversion_threads`の違い

| 設定 | 役割 | 初心者向けの目安 |
|---|---|---|
|`threads`|codec内部のdecode/encode並列度|`0`で自動。再現比較では`1`|
|`prefetch`|decode済みframeを先に用意するqueue長|`0`で同期、CPU通常処理は`4`程度|
|`conversion_threads`|I420からBGR/RGB/BGRAへの色変換thread総数|`0`で自動、`1`で補助workerなし|
|`queue_size`|encode入力を保持するbounded queue長|`0`で同期、CPU通常処理は`4～8`|

これらは別々の仕事を制御します。値を増やすほど必ず速くなるわけではありません。GPU-resident surfaceは現在`prefetch=0`、`queue_size=0`を使用します。

## 8. borrowed NumPy view

通常の`read()`は扱いやすいowned BGR配列です。`read_borrowed()`はnative I420 bufferをcopyせずread-only NumPy viewとして借ります。高速ですが、viewの生存中はnative bufferを再利用できず、内容を書き換えてはいけません。まず通常の`read()`を使い、copyがボトルネックだと測定できた場合に選びましょう。

In [ ]:
with mkvcodec.VideoCapture(source_video, prefetch=0) as capture:
    borrowed = capture.read_borrowed()
    if borrowed is not None:
        with borrowed as frame:
            y, u, v = frame.planes
            print('Y/U/V shapes:', y.shape, u.shape, v.shape)
            print('writeable     :', y.flags.writeable, u.flags.writeable, v.flags.writeable)
            print('PTS(ns)       :', frame.pts_ns)

## 9. batch read

`read_batch()`は指定数までをpresentation順にまとめて返します。最後のbatchはEOSやtimeoutで短くなります。CPU frameはownedなので、次のbatchを読んだ後も保持できます。

In [ ]:
batch_sizes = []
with mkvcodec.VideoCapture(source_video, prefetch=4) as capture:
    while batch := capture.read_batch(16, timeout_ms=100, format='bgr'):
        batch_sizes.append(len(batch))
print('batch sizes:', batch_sizes, 'total:', sum(batch_sizes))

## 10. metricsを読む

`metrics`はframe数、queue待機、backend時間、実際のcopy pathを返します。`copy_edge_metrics`はlibraryが制御する境界でのshared surface、GPU copy、CPU upload/readback、layout正規化、pixel変換を分けます。vendor driver内部は観測できないため、counterが0でもdriver全体の完全なzero-copy証明にはなりません。

In [ ]:
with mkvcodec.VideoCapture(source_video, prefetch=0) as capture:
    while capture.read() is not None:
        pass

print('pipeline metrics:', capture.metrics)
print('stage metrics   :', capture.stage_metrics)
print('component       :', capture.component_metrics)
print('copy edges      :', capture.copy_edge_metrics)

## 11. GPU-resident経路（対応環境だけ実行）

GPUでは`read_surface()`がNumPyではなく`GpuFrame` leaseを返します。そのまま同じbackendのWriterへ渡すか、native handle/DLPackを外部GPU libraryへ渡します。`require_gpu_resident=True`はCPU stagingが必要な操作を拒否します。

次のセルは、VP9 decodeとAV1 encodeの両方を同じGPU backendが提供するときだけ実行します。該当GPUがなければ安全にskipします。

In [ ]:
def supports(backend: str, codec: str, direction: str) -> bool:
    return any(
        item.backend == backend
        and item.codec == codec
        and item.is_hardware
        and (item.can_decode if direction == 'decode' else item.can_encode)
        for item in mkvcodec.backend_capabilities()
    )

gpu_backend = next(
    (name for name in ('nvidia', 'intel')
     if supports(name, 'vp9', 'decode') and supports(name, 'av1', 'encode')),
    None,
)

if gpu_backend is None:
    print('VP9 decode→AV1 encodeに対応するGPUがないためskipします。')
else:
    gpu_output = output_dir / f'tutorial_{gpu_backend}_gpu.webm'
    with mkvcodec.VideoCapture(
        source_video, codec='vp9', backend=gpu_backend,
        prefetch=0, require_gpu_resident=True,
    ) as capture:
        with mkvcodec.VideoWriter(
            gpu_output, codec='av1', backend=gpu_backend,
            fps=info.fps or 30, frame_size=(info.width, info.height),
            queue_size=0, require_gpu_resident=True,
        ) as writer:
            gpu_frames = 0
            while (surface := capture.read_surface()) is not None:
                with surface:
                    if gpu_frames == 0:
                        print('interop:', surface.interop)
                    writer.write_surface(surface)
                    gpu_frames += 1
    print('GPU output:', gpu_output, 'frames:', gpu_frames)
    print('decode path:', capture.metrics.copy_path)
    print('encode path:', writer.metrics.copy_path)

### 外部GPU画像処理を挟む場合

上の例はsurfaceを直接encodeしました。画像処理を挟む場合は`surface.interop`を確認し、環境に合うadapterを使います。

- NVIDIA: CUDA pointer / CUDA array / DLPack → CuPy、NPP、CUDA kernel
- Intel Windows: D3D11 texture/fence → D3D11 compute、DirectML等
- Intel Linux: VA surface / OpenCL / SYCL / device-USM

処理済みresourceをimportするときはownerとproducer event/fenceを渡します。Intel device-USMはv0.1 Previewで、`frame.interop.api_stability`から確認できます。pointerのcontext/device provenanceはcaller側oneAPI runtimeで検証してください。

## 12. よくあるエラー

| 症状 | 確認すること |
|---|---|
|`Unable to load mkvcodec native library`|`MKVC_LIBRARY_PATH`が実在するDLL/soを指すか、依存libraryが見つかるか|
|`no matching ... backend is available`|`backend_capabilities()`でcodecとdecode/encode方向を確認|
|GPU指定でCPU readが失敗|`require_gpu_resident=True`では`read()`でなく`read_surface()`を使う|
|GPU surfaceとWriterが合わない|同じbackend/deviceか、出力codecをGPUがencode可能か確認|
|配列layoutが拒否される|dtype=`uint8`、偶数寸法、shape、stride、`strict_cpu_layout`を確認|
|処理が速くならない|`threads`、`prefetch`、色変換、画像処理、encodeのどこが律速かmetricsで分ける|

decoder seekはv0.1では未対応です。動画は先頭からEOSまで順番に読みます。

## 13. 練習課題

1. `make_frame()`の色や移動方向を変えてください。
2. 処理セルを上下反転（`frame[::-1]`）やcropへ変更してください。
3. `queue_size=0`と`4`、`prefetch=0`と`4`を同じPCで測ってください。
4. VP9 Writerを`codec='av1'`へ変え、`probe_video()`の結果を確認してください。
5. OpenCVがある場合は`cv2.putText`や`cv2.resize`を処理loopへ追加してください。

さらに詳しく知りたい場合は、[README](../README.md)、[外部仕様](../docs/external-spec/system-spec.md)、[性能測定](../docs/benchmarking.md)、[テスト仕様](../docs/test-spec/test-requirements.md)を参照してください。